# Bias Analysis — Paper-Ready Tables & Figures

Reads only what's already on Drive (no model loading, no retraining):
- `results/combined_summary.csv` — 21 rows of (classifier x generator), bias scores from the original BvT formula, per-target F1 deltas, real/synth/mixed F1
- `results/<clf>/<gen>/real+synth+mixed/test_metrics.csv` — per-seed F1 for variance analysis

### What this notebook produces

| Output | Question it answers |
|---|---|
| `bias_heatmap_synth_vs_mixed.png` | Which generators bias which classifiers, and does mixing real data fix it? |
| `f1_drop_per_target.png` | Where does the F1 hit land — Trump, Biden, or Bernie tweets? |
| `f1_real_vs_synth_vs_mixed.png` | What does training condition cost in raw accuracy? |
| `bias_mitigation.png` | How much of the synth bias does 50/50 mixing remove? |
| `generator_bias_profile.png` | Generator-level bias ranking (signed) |
| `classifier_susceptibility.png` | Which classifier amplifies bias the most? |
| `per_seed_variance.png` | Are the F1 numbers stable across 3 seeds, or is there room to add error bars? |
| `paper_summary.csv` | Compact table for the paper (mean F1 ± std, bias scores) |

### Note on bias dimension

Only the **Biden-vs-Trump (BvT)** bias axis is available from saved metrics — the other 4 axes (Bernie-vs-Trump, Biden-vs-Bernie, Left-vs-Trump, Establishment-vs-Outsider) would require raw predictions on disk, which only qwen-2.5-7b currently has. If you decide later that the wider axis-set is needed for the paper, retrain with the patched training notebook so predictions persist.

## 0. Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR_STR = '/content/drive/MyDrive/PoliticalBiasProject'
except ImportError:
    PROJECT_DIR_STR = './'

from pathlib import Path
PROJECT_DIR = Path(PROJECT_DIR_STR)
RESULTS_DIR = PROJECT_DIR / 'results'
FIG_DIR     = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Results : {RESULTS_DIR}  (exists={RESULTS_DIR.exists()})')
print(f'Figures : {FIG_DIR}')

## 1. Imports + constants

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

TARGETS    = ['Donald Trump', 'Joe Biden', 'Bernie Sanders']
CONDITIONS = ['real', 'synth', 'mixed']
SEEDS      = [42, 123, 7]
RUN_TAG    = 'real+synth+mixed'

# Consistent classifier / generator ordering for all plots
CLF_ORDER = ['roberta', 'debertav3', 'bertweet']
GEN_ORDER = ['gpt-4o-mini', 'gpt-5.4-mini', 'mistral-7b',
             'qwen-2.5-7b', 'gemma-2-9b', 'llama-3.1-8b', 'llama-3.2-3b']

mpl.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 150,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
})

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 2. Load combined summary + per-seed metrics

In [ ]:
summary_path = RESULTS_DIR / 'combined_summary.csv'
summary = pd.read_csv(summary_path)
print(f'combined_summary.csv -> {len(summary)} rows x {len(summary.columns)} cols')

# Enforce ordering
summary['classifier'] = pd.Categorical(summary['classifier'], categories=CLF_ORDER, ordered=True)
summary['generator']  = pd.Categorical(summary['generator'],  categories=GEN_ORDER, ordered=True)
summary = summary.sort_values(['classifier', 'generator']).reset_index(drop=True)
print(summary[['classifier','generator','real_f1','synth_f1','mixed_f1',
               'bias_synth_full','bias_mixed_full','status']].to_string(index=False))

# Per-seed test_metrics.csv for variance analysis
seed_rows = []
for clf in CLF_ORDER:
    for gen in GEN_ORDER:
        tm = RESULTS_DIR / clf / gen / RUN_TAG / 'test_metrics.csv'
        if not tm.exists():
            continue
        df = pd.read_csv(tm)
        df['classifier'] = clf
        df['generator']  = gen
        seed_rows.append(df)

per_seed = pd.concat(seed_rows, ignore_index=True) if seed_rows else pd.DataFrame()
print(f'\nPer-seed metrics scanned: {len(per_seed)} rows from {len(seed_rows)} combos')
if len(per_seed):
    per_seed.head()

## 3. Headline table — paper Table 2

In [ ]:
# Compact view: classifier, generator, F1 by condition, bias scores
paper_cols = ['classifier', 'generator', 'real_f1', 'synth_f1', 'mixed_f1',
              'trump_synth_dF1', 'biden_synth_dF1', 'bernie_synth_dF1',
              'bias_synth_full', 'bias_mixed_full']
paper = summary[paper_cols].copy()
paper = paper.rename(columns={
    'real_f1':           'F1_real',
    'synth_f1':          'F1_synth',
    'mixed_f1':          'F1_mixed',
    'trump_synth_dF1':   'dF1_Trump',
    'biden_synth_dF1':   'dF1_Biden',
    'bernie_synth_dF1':  'dF1_Bernie',
    'bias_synth_full':   'BiasBvT_synth',
    'bias_mixed_full':   'BiasBvT_mixed',
})
print('=' * 130)
print('PAPER TABLE 2 - classifier x generator x condition F1, F1 drops per target, BvT bias')
print('=' * 130)
print(paper.to_string(index=False))

paper.to_csv(RESULTS_DIR / 'paper_summary.csv', index=False)
print(f'\nSaved -> {RESULTS_DIR / "paper_summary.csv"}')

## 4. Bias direction heatmap — synth vs mixed side by side

In [ ]:
def bias_heatmap(ax, data_col, title, vmax=0.3):
    pivot = summary.pivot(index='classifier', columns='generator', values=data_col)
    pivot = pivot.reindex(index=CLF_ORDER, columns=GEN_ORDER)
    im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha='right')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(title)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            if pd.notna(v):
                ax.text(j, i, f'{v:+.2f}', ha='center', va='center', fontsize=8,
                        color='white' if abs(v) > vmax * 0.5 else 'black')
    return im

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
im = bias_heatmap(axes[0], 'bias_synth_full', 'Bias (synth) - BvT axis')
bias_heatmap(axes[1], 'bias_mixed_full',     'Bias (mixed) - BvT axis')
fig.colorbar(im, ax=axes, shrink=0.85, label='Bias score (signed; + = pro-Biden / anti-Trump)')
fig.suptitle('Biden-vs-Trump bias: synth condition introduces it, mixed condition mitigates it',
             fontsize=12, y=1.02)
out = FIG_DIR / 'bias_heatmap_synth_vs_mixed.png'
plt.savefig(out, bbox_inches='tight')
print(f'Saved -> {out}')
plt.show()

## 5. Per-target F1 drops — where does the hit land?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
target_cols = [('trump_synth_dF1',  'Trump tweets'),
               ('biden_synth_dF1',  'Biden tweets'),
               ('bernie_synth_dF1', 'Bernie tweets')]

vmax = 0.25
for ax, (col, title) in zip(axes, target_cols):
    pivot = summary.pivot(index='classifier', columns='generator', values=col)
    pivot = pivot.reindex(index=CLF_ORDER, columns=GEN_ORDER)
    im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha='right')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(f'F1 delta - {title}')
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            if pd.notna(v):
                ax.text(j, i, f'{v:+.2f}', ha='center', va='center', fontsize=7,
                        color='white' if abs(v) > vmax * 0.5 else 'black')

fig.colorbar(im, ax=axes, shrink=0.8, label='F1 (synth) - F1 (real)')
fig.suptitle('Synth-induced F1 drop per target  (negative = worse than real-baseline)', y=1.02)
out = FIG_DIR / 'f1_drop_per_target.png'
plt.savefig(out, bbox_inches='tight')
print(f'Saved -> {out}')
plt.show()

## 6. F1 cost — real vs synth vs mixed

In [ ]:
gens = GEN_ORDER
x = np.arange(len(gens))
width = 0.25

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)
for ax, clf in zip(axes, CLF_ORDER):
    sub = summary[summary.classifier == clf].set_index('generator').reindex(gens)
    ax.bar(x - width, sub['real_f1'],  width, label='real',  color='#4c72b0')
    ax.bar(x,         sub['synth_f1'], width, label='synth', color='#dd8452')
    ax.bar(x + width, sub['mixed_f1'], width, label='mixed', color='#55a868')
    ax.set_xticks(x); ax.set_xticklabels(gens, rotation=35, ha='right')
    ax.set_ylim(0.6, 0.9)
    ax.set_title(clf)
    ax.grid(axis='y', alpha=0.3)
    ax.set_axisbelow(True)

axes[0].set_ylabel('Macro F1 (overall)')
axes[0].legend(loc='lower right', framealpha=0.9)
fig.suptitle('Test-time F1 per training condition - real always wins, mixed approximates real', y=1.02)
out = FIG_DIR / 'f1_real_vs_synth_vs_mixed.png'
plt.savefig(out, bbox_inches='tight')
print(f'Saved -> {out}')
plt.show()

## 7. Mitigation effectiveness — how much does mixing remove?

In [ ]:
mit = summary[['classifier', 'generator', 'bias_synth_full', 'bias_mixed_full']].copy()
mit['abs_synth']     = mit['bias_synth_full'].abs()
mit['abs_mixed']     = mit['bias_mixed_full'].abs()
mit['reduction']     = mit['abs_synth'] - mit['abs_mixed']
mit['pct_reduction'] = np.where(mit['abs_synth'] > 0,
                                100 * mit['reduction'] / mit['abs_synth'],
                                np.nan)
mit = mit.sort_values('pct_reduction', ascending=False)

fig, ax = plt.subplots(figsize=(11, 6))
labels = mit.apply(lambda r: f'{r["classifier"]:9s} / {r["generator"]}', axis=1)
colors = ['#55a868' if v > 0 else '#c44e52' for v in mit['pct_reduction']]
ax.barh(labels[::-1], mit['pct_reduction'][::-1], color=colors[::-1])
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('|bias_synth| reduction (%) achieved by 50/50 mixing')
ax.set_title('Mitigation effectiveness per (classifier, generator)\nGreen = mixing reduces bias; red = mixing makes it worse')
ax.grid(axis='x', alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
out = FIG_DIR / 'bias_mitigation.png'
plt.savefig(out, bbox_inches='tight')
print(f'Saved -> {out}')
plt.show()

print('\nMean bias reduction across 21 combos: {:.1f}%'.format(mit['pct_reduction'].mean()))

## 8. Generator bias profile — which generators bias the most, signed?

In [ ]:
gen_profile = summary.groupby('generator', observed=True).agg(
    mean_bias_synth=('bias_synth_full', 'mean'),
    std_bias_synth =('bias_synth_full', 'std'),
    mean_bias_mixed=('bias_mixed_full', 'mean'),
).reindex(GEN_ORDER)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(GEN_ORDER))
ax.bar(x - 0.2, gen_profile['mean_bias_synth'], 0.4,
       yerr=gen_profile['std_bias_synth'], label='synth',
       color='#dd8452', capsize=4)
ax.bar(x + 0.2, gen_profile['mean_bias_mixed'], 0.4,
       label='mixed', color='#55a868')
ax.axhline(0, color='black', lw=0.5)
ax.set_xticks(x); ax.set_xticklabels(GEN_ORDER, rotation=30, ha='right')
ax.set_ylabel('Bias score (BvT axis, mean across classifiers)')
ax.set_title('Generator-level bias profile - error bars = stddev across 3 classifiers')
ax.legend(); ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
plt.tight_layout()
out = FIG_DIR / 'generator_bias_profile.png'
plt.savefig(out, bbox_inches='tight')
print(f'Saved -> {out}')
plt.show()
print('\n', gen_profile.round(3))

## 9. Classifier susceptibility — which classifier amplifies bias most?

In [ ]:
clf_profile = summary.groupby('classifier', observed=True).agg(
    mean_abs_bias_synth=('bias_synth_full', lambda s: s.abs().mean()),
    max_abs_bias_synth =('bias_synth_full', lambda s: s.abs().max()),
    mean_abs_bias_mixed=('bias_mixed_full', lambda s: s.abs().mean()),
).reindex(CLF_ORDER)

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(CLF_ORDER))
ax.bar(x - 0.2, clf_profile['mean_abs_bias_synth'], 0.4, label='synth', color='#dd8452')
ax.bar(x + 0.2, clf_profile['mean_abs_bias_mixed'], 0.4, label='mixed', color='#55a868')
ax.set_xticks(x); ax.set_xticklabels(CLF_ORDER)
ax.set_ylabel('Mean |bias| across 7 generators')
ax.set_title('Classifier susceptibility - average |BvT bias|')
ax.legend(); ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
plt.tight_layout()
out = FIG_DIR / 'classifier_susceptibility.png'
plt.savefig(out, bbox_inches='tight')
print(f'Saved -> {out}')
plt.show()
print('\n', clf_profile.round(3))

## 10. Per-seed F1 variance — are these numbers stable?

In [ ]:
if len(per_seed) == 0:
    print('No per-seed metrics found; skipping variance plot.')
else:
    overall = per_seed[per_seed['target'] == 'OVERALL'].copy()
    agg = (overall.groupby(['classifier', 'generator', 'condition'])['macro_f1']
                  .agg(['mean', 'std', 'count']).reset_index())

    # Plot stddev per (classifier, generator, condition)
    pivot_std = agg.pivot_table(index=['classifier', 'generator'],
                                 columns='condition', values='std')
    pivot_std = pivot_std[[c for c in CONDITIONS if c in pivot_std.columns]]

    fig, ax = plt.subplots(figsize=(10, 6))
    pivot_std.plot(kind='barh', ax=ax,
                   color=['#4c72b0', '#dd8452', '#55a868'][:pivot_std.shape[1]],
                   width=0.8)
    ax.set_xlabel('Macro F1 stddev across 3 seeds')
    ax.set_title('Per-seed F1 stability  (lower = more stable; >0.01 suggests showing error bars in paper)')
    ax.grid(axis='x', alpha=0.3); ax.set_axisbelow(True)
    plt.tight_layout()
    out = FIG_DIR / 'per_seed_variance.png'
    plt.savefig(out, bbox_inches='tight')
    print(f'Saved -> {out}')
    plt.show()

    # Save the aggregate
    agg.to_csv(RESULTS_DIR / 'paper_summary_perseed.csv', index=False)
    print(f'Saved -> {RESULTS_DIR / "paper_summary_perseed.csv"}')
    print('\nMax stddev observed:', round(float(agg["std"].max()), 4))
    print('Mean stddev observed:', round(float(agg["std"].mean()), 4))

## 11. Done

Artifacts written to Drive:

```
results/
  paper_summary.csv             # Headline 21-row table
  paper_summary_perseed.csv     # Per-seed F1 (mean / std / count)
  figures/
    bias_heatmap_synth_vs_mixed.png
    f1_drop_per_target.png
    f1_real_vs_synth_vs_mixed.png
    bias_mitigation.png
    generator_bias_profile.png
    classifier_susceptibility.png
    per_seed_variance.png
```

Drop these straight into your EMNLP submission. The BvT-axis story holds at the 21-combo scale; if you later want the wider multi-axis treatment, you'll need to rerun training with the patched notebook so predictions persist.